# Comparing Safety-Removal Subspaces in Aligned LLMs

This notebook reproduces the experiments for the course-project report on safety-removal subspaces in Llama-3.1-8B-Instruct. It runs the three methods compared in the report, evaluates their safety/utility tradeoff, computes the geometric analyses, and packages the resulting figures and JSON files.

## Methods

| Method | Artifact | Intervention |
|---|---|---|
| DIM | one residual-stream direction | inference-time 1-D ablation |
| ActSVD | modified model weights | permanent weight-space edit |
| RCO | 2-D refusal cone | inference-time 2-D subspace ablation |

## Analysis Outputs

| Step | Main output |
|---|---|
| Behavioral benchmark | `code/results/benchmark/benchmark_results.json` |
| Safety-utility overlap | `code/results/safety_utility_overlap/safety_utility_overlap_results.json` |
| Method overlap | `code/results/method_overlap/comparison_results.json` |
| RepInd | `code/results/geometry_repind_rco/geometry_repind_results.json` |
| Prompt probe | `code/results/probe_attack_types/results.json` |
| Report figures | `docs/figures/*.png` |

The main overlap metric is normalized MSO:

$$\text{MSO}(A, B) = \frac{\| U_A^T U_B \|_F^2}{\min(k_A, k_B)}$$

with random baseline $\max(k_A, k_B) / d$.

## Runtime

Use an A100 runtime for the full Llama-3.1-8B-Instruct run. Llama and WildJailbreak require a Hugging Face token with access to the relevant gated resources.


In [ ]:
from pathlib import Path
import os, subprocess

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

REPO_URL = "https://github.com/evan203/nlp-project-proposal.git"
BRANCH = "experiment/kyle-v2"
PROJECT = Path('/content/nlp-project-proposal')

# --- Model selection ---
# For the full report: meta-llama/Llama-3.1-8B-Instruct (requires A100 + HF token)
# Smaller model option: google/gemma-2b-it (no token needed, T4-compatible)
MODEL_PATH = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_ALIAS = MODEL_PATH.split('/')[-1]  # e.g. Llama-3.1-8B-Instruct

HF_HOME = PROJECT / 'code' / 'llm_weights'
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME)
os.environ['PYTHON_RUNNER'] = 'python'

def run(cmd, cwd=None, env_extra=None):
    """Run a shell command and stream output line-by-line."""
    env = os.environ.copy()
    if env_extra:
        env.update({k: str(v) for k, v in env_extra.items()})
    effective_cwd = cwd if cwd is not None else (PROJECT if PROJECT.exists() else None)
    print(f"\n$ {cmd}")
    with subprocess.Popen(
        cmd, shell=True, cwd=effective_cwd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end='')
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, cmd)

print('GPU status:')
run('nvidia-smi')


## 1. Clone the repository and authenticate Hugging Face

This cell clones the project, checks out the working branch, and authenticates Hugging Face. Authentication is required for `meta-llama/Llama-3.1-8B-Instruct` and for WildJailbreak streaming in the probe step.


In [ ]:
if not PROJECT.exists():
    run(f"git clone -b {BRANCH} {REPO_URL} {PROJECT}", cwd="/content")
else:
    run(f"git checkout {BRANCH} && git pull", cwd=PROJECT)

run("pip -q install -U huggingface_hub")

from huggingface_hub import notebook_login, whoami

try:
    user = whoami()
    print("Already logged into Hugging Face as:", user["name"])
except Exception:
    notebook_login()


## 2. Install dependencies

Install the packages needed by the method pipelines, benchmark evaluation, plotting scripts, and model hooks. `nnsight==0.3.7` is pinned for compatibility with the cones/RCO code.


In [ ]:
run('pip -q install -U torch torchvision transformers accelerate datasets '
    'sentencepiece protobuf tqdm jaxtyping matplotlib seaborn '
    'zstandard litellm einops safetensors')
run('pip -q install -U nnsight==0.3.7')
run('pip -q install -U vllm')  # required by evaluate_jailbreak.py imports
run('pip -q install python-dotenv wandb')  # required by RCO training

# Verify GPU is accessible
run("python -c 'import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))'")  # noqa


## 3. Run DIM

DIM computes harmful-minus-harmless mean activation differences, selects the report direction, and saves the direction artifacts used by later steps.

Main outputs:
- `code/methods/dim/pipeline/runs/<model>/direction.pt`
- `code/methods/dim/pipeline/runs/<model>/direction_metadata.json`
- `code/methods/dim/pipeline/runs/<model>/generate_directions/mean_diffs.pt`


In [ ]:
run('./scripts/run_dim.sh', env_extra={'MODEL_PATH': MODEL_PATH})


## 4. Run ActSVD

ActSVD performs weight-space safety removal using the report hyperparameters `RANK_POS=3950`, `RANK_NEG=4090`, and `NSAMPLES=128`.

Main output:
- `code/methods/actsvd/out/`


In [ ]:
# ActSVD with paper-optimal hyperparams (Wei et al. 2024 Table 6 best config).
# Their optimal: r^u=3950 (utility), r^s=4090 (safety), effective Delta-W rank ~6.
run('./scripts/run_actsvd.sh', env_extra={
    'MODEL_ALIAS': MODEL_ALIAS,
    'NSAMPLES':    '128',
    'RANK_POS':    '3950',  # utility rank r^u  (paper optimal)
    'RANK_NEG':    '4090',  # safety rank  r^s  (paper optimal)
    'EVAL_PPL':    '0',
    'EVAL_ATTACK': '0',
})


## 5. Run RCO

RCO trains a 2-D refusal cone initialized from the DIM direction. The evaluation uses true 2-D subspace ablation: both orthonormal cone basis vectors are projected out.

Main outputs:
- `code/results/geometry_repind/rco_direction.pt`
- `code/results/geometry_repind/directions_rco.json`
- `code/results/geometry_repind_rco/geometry_repind_results.json`
- RCO benchmark entries in `code/results/benchmark/benchmark_results.json`


In [ ]:
run('./scripts/run_rco_eval.sh', env_extra={
    'MODEL': MODEL_PATH,
    'MODE': 'cone',
    'CONE_DIM': '2',
    'METHOD_NAME': 'RCO-Cone-2',
    'WANDB_MODE': 'offline',
    'SKIP_EVAL': '0',      # eval_direction_benchmark.py runs inside this script
    'SKIP_REPIND': '0',
    'SAVE_MODIFIED': '0',
})


## 6. Behavioral Benchmark

Evaluate the base model, DIM, ActSVD, RCO, and random baselines on JailbreakBench, harmless compliance, perplexity, and TruthfulQA. The substring ASR is computed here; the Qwen3Guard judge is applied in the next step after all completions are saved.

Main output:
- `code/results/benchmark/benchmark_results.json`


In [ ]:
import os
CODE = PROJECT / 'code'
DIM_DIR_RUN = CODE / f'methods/dim/pipeline/runs/{MODEL_ALIAS}'

# Common args appended to every eval invocation:
#   --eval_ppl --n_ppl_samples 64           (perplexity)
#   --eval_truthfulqa --n_tqa_samples 64    (TruthfulQA side-effect test)
#   --bootstrap 1000                         (95% CI on ASR + harmless compliance)
# NOTE: no judge runs here. Qwen3Guard is run as a separate post-hoc
# pass in step 6b after all method completions are saved.
COMMON_ARGS = (
    ' --eval_ppl --n_ppl_samples 64'
    ' --eval_truthfulqa --n_tqa_samples 64'
    ' --bootstrap 1000'
)

# Base model -- no intervention
run(
    f'python analysis/eval_direction_benchmark.py'
    f' --model_path {MODEL_PATH}'
    f' --method_name "Base ({MODEL_ALIAS})"'
    f' --no_ablation' + COMMON_ARGS,
    cwd=CODE,
)

# DIM -- 1-D direction ablation
run(
    f'python analysis/eval_direction_benchmark.py'
    f' --model_path {MODEL_PATH}'
    f' --direction_path {DIM_DIR_RUN}/direction.pt'
    f' --direction_metadata {DIM_DIR_RUN}/direction_metadata.json'
    f' --method_name DIM-Ablated' + COMMON_ARGS,
    cwd=CODE,
)

# ActSVD -- load modified model directly (no direction hooks)
run(
    f'python analysis/eval_direction_benchmark.py'
    f' --model_path {MODEL_PATH}'
    f' --modified_model_path methods/actsvd/out'
    f' --method_name ActSVD-Modified' + COMMON_ARGS,
    cwd=CODE,
)

# Random 1-D direction baseline -- sanity check that ablating an arbitrary
# unit vector does NOT raise ASR. Matches DIM's 1-D intervention rank.
run(
    f'python analysis/eval_direction_benchmark.py'
    f' --model_path {MODEL_PATH}'
    f' --random_direction --random_subspace_dim 1 --seed 7'
    f' --method_name Random-Direction-7-1D' + COMMON_ARGS,
    cwd=CODE,
)

# Random 2-D subspace baseline -- matches RCO's cone dim. Expected ASR
# again near base; rules out "removing more dimensions inflates ASR" as
# an explanation for RCO's high ASR.
run(
    f'python analysis/eval_direction_benchmark.py'
    f' --model_path {MODEL_PATH}'
    f' --random_direction --random_subspace_dim 2 --seed 7'
    f' --method_name Random-Subspace-7-2D' + COMMON_ARGS,
    cwd=CODE,
)

# Show current state of benchmark_results.json
import json, pathlib
benchmark_path = CODE / 'results/benchmark/benchmark_results.json'
if benchmark_path.exists():
    benchmark = json.loads(benchmark_path.read_text())
    print('\nbenchmark_results.json entries (substring judge -- Qwen3Guard added in step 6b):')
    for name, entry in benchmark.items():
        jbb = entry.get('jailbreakbench', {})
        asr = jbb.get('asr', 'N/A')
        hc  = entry.get('harmless_compliance', {}).get('rate', 'N/A')
        tqa = entry.get('truthfulqa', {}).get('truthful_rate')
        ppl_pile = entry.get('perplexity', {}).get('pile', {}).get('perplexity', 'N/A')
        suffix = f', tqa={tqa}' if tqa is not None else ''
        print(f'  {name}: ASR={asr}, harmless={hc}, ppl_pile={ppl_pile}{suffix}')


### 6b. Post-hoc Qwen3Guard Judge

Grade all saved JailbreakBench completions with `Qwen/Qwen3Guard-Gen-4B`. The report defines jailbreak success as the `Unsafe` label; `Controversial` is counted as not jailbroken unless `--include_controversial` is set.


In [ ]:
run(
    f'python analysis/judge_completions.py'
    f' --judge_model_path Qwen/Qwen3Guard-Gen-4B'
    f' --benchmark_dir results/benchmark'
    f' --batch_size 8'
    f' --bootstrap 1000',
    cwd=CODE,
)

# Show updated benchmark_results.json with both judges
import json
benchmark_path = CODE / 'results/benchmark/benchmark_results.json'
if benchmark_path.exists():
    benchmark = json.loads(benchmark_path.read_text())
    print('\nbenchmark_results.json after Qwen3Guard judge:')
    print(f'{"Method":<32} {"sub-ASR":>8} {"Q3G-ASR":>8} {"harmless":>9} {"ppl_pile":>9}')
    for name, entry in benchmark.items():
        jbb = entry.get('jailbreakbench', {})
        sub = jbb.get('asr')
        llm = jbb.get('asr_llm_judge')
        hc  = entry.get('harmless_compliance', {}).get('rate')
        ppl = entry.get('perplexity', {}).get('pile', {}).get('perplexity')
        sub_s = f'{sub:.2f}' if isinstance(sub, (int,float)) else str(sub)
        llm_s = f'{llm:.2f}' if isinstance(llm, (int,float)) else 'N/A'
        hc_s  = f'{hc:.2f}'  if isinstance(hc,  (int,float)) else str(hc)
        ppl_s = f'{ppl:.2f}' if isinstance(ppl, (int,float)) else str(ppl)
        print(f'{name:<32} {sub_s:>8} {llm_s:>8} {hc_s:>9} {ppl_s:>9}')


## 7. Safety-Utility Activation Overlap

Compute utility PCA bases from harmless activations and measure how much each method's safety object overlaps those utility subspaces. RCO is evaluated with normalized 2-D subspace MSO.

Main outputs:
- `code/results/safety_utility_overlap/safety_utility_overlap_results.json`
- `code/results/safety_utility_overlap/safety_utility_overlap_per_layer.png`
- `code/results/safety_utility_overlap/safety_utility_overlap_by_rank.png`


In [ ]:
run('./scripts/run_safety_utility_overlap.sh', env_extra={
    'MODEL_PATH': MODEL_PATH,
    'N_UTILITY_SAMPLES': '128',
    'UTILITY_RANKS': '1,2,4,8,16,32',
    'PRIMARY_RANK': '8',
    'ACTSVD_OUT': str(CODE / 'methods/actsvd/out'),
})


## 8. Method Subspace Overlap

Compare activation-space directions from DIM/RCO to the ActSVD weight-delta column space using per-layer MSO. This is a CPU-only analysis over saved model weights and direction tensors.

Main outputs:
- `code/results/method_overlap/comparison_results.json`
- `code/results/method_overlap/mso_per_layer.png`
- `code/results/method_overlap/cross_model_cosine.png`


In [ ]:
run('./scripts/run_method_overlap.sh', env_extra={
    'MODEL_PATH': MODEL_PATH,
    'ACTSVD_MODEL_PATH': str(CODE / 'methods/actsvd/out'),
})


## 9. RepInd Profile Analysis

Run the representational-independence comparison for DIM and RCO directions. This measures how ablating one direction changes another direction's per-layer cosine profile.

Main outputs:
- `code/results/geometry_repind_rco/geometry_repind_results.json`
- `code/results/geometry_repind_rco/repind_change_heatmap.png`


In [ ]:
run('./scripts/run_geometry_repind.sh', env_extra={
    'MODEL_PATH': MODEL_PATH,
    'N_PROMPTS': '32',
    'BATCH_SIZE': '8',
    'DERIVED_CONE_DIM': '3',
})


## 10. Prompt-Attack Probe

Test whether prompt-level jailbreaks suppress the same refusal geometry that DIM and RCO ablate. The probe uses HarmBench direct requests, WildJailbreak adversarial harmful prompts, and WildJailbreak adversarial benign controls.

Main outputs:
- `code/results/probe_attack_types/results.json`
- `code/results/probe_attack_types/layer_sweep.json`
- probe figures in `code/results/probe_attack_types/`


In [ ]:
run('./scripts/run_probe_attack_types.sh', env_extra={
    'MODEL_PATH':       MODEL_PATH,
    'MODEL_ALIAS':      MODEL_ALIAS,
    'N_DIRECT':         '25',   # HarmBench direct-request baseline prompts
    'N_PER_TYPE':       '25',   # WildJailbreak prompts per group
    'BATCH_SIZE':       '8',
    'MAX_NEW_TOKENS':   '200',
    'LAYER_SWEEP':      '1',     # 1 = per-layer projection sweep
    'ABLATION_CROSS':   '1',     # 1 = generate completions with each method ablated
    'ABLATION_METHODS': 'auto',  # 'auto' = ablate every loaded direction (DIM 1-D + RCO 2-D cone)
    'BOOTSTRAP':        '1000',  # 0 = disable bootstrap CIs
})

# Print per-group summary, including per-method ablation cross-tests.
import json as _json
from collections import defaultdict as _dd
probe_path = CODE / 'results/probe_attack_types/results.json'
if probe_path.exists():
    _results = _json.loads(probe_path.read_text())
    for _r in _results:
        if 'tactic' in _r and 'attack_type' not in _r:
            _r['attack_type'] = _r['tactic']

    _proj_keys = sorted({k for _r in _results for k in _r
                         if k.startswith('proj_') and '_basis_' not in k})
    _abl_keys = sorted({k.replace('ablated_is_jailbreak_', '')
                        for _r in _results for k in _r
                        if k.startswith('ablated_is_jailbreak_')})
    _by_type = _dd(list)
    for _r in _results:
        _by_type[_r['attack_type']].append(_r)

    _hdr = f'{"Group":<26} {"n":>3} {"ASR":>5}'
    for _m in _abl_keys:
        _hdr += f'  {f"AblASR_{_m}":>14}'
    for _pk in _proj_keys:
        _dn = _pk.replace('proj_', '')
        _hdr += f'  {f"proj_{_dn}":>16}'
    print(_hdr)
    print('-' * len(_hdr))

    _order = ['direct_request', 'adversarial_harmful', 'adversarial_benign']
    for _atype in _order:
        if _atype not in _by_type:
            continue
        _entries = _by_type[_atype]
        _n = len(_entries)
        _asr = sum(e['is_jailbreak'] for e in _entries) / _n
        _row = f'{_atype:<26} {_n:>3} {_asr:>5.2f}'
        for _m in _abl_keys:
            _key = f'ablated_is_jailbreak_{_m}'
            _abl = sum(e.get(_key, 0) for e in _entries) / _n
            _row += f'  {_abl:>14.2f}'
        for _pk in _proj_keys:
            _mp = sum(e.get(_pk, 0) for e in _entries) / _n
            _row += f'  {_mp:>16.4f}'
        print(_row)


## 11. Regenerate Figures and Sync to `docs/figures`

Rerun plotting scripts from the saved JSON files and copy report-facing PNGs into `docs/figures/`.


In [ ]:
import pathlib

CODE_DIR = PROJECT / 'code'

# Benchmark: all methods now in benchmark_results.json
run('python analysis/plot_benchmarks.py', cwd=CODE_DIR)

# Method subspace overlap: dynamic, plots all pairs found in comparison_results.json
run('python analysis/plot_method_overlap.py', cwd=CODE_DIR)

# Safety-utility overlap: DIM mean-diffs + overlaid RCO/DIM direction lines
run('python analysis/plot_safety_utility_overlap.py', cwd=CODE_DIR)

# RepInd: use RCO results if available, otherwise DIM-derived
rco_repind = CODE_DIR / 'results/geometry_repind_rco/geometry_repind_results.json'
if rco_repind.exists():
    run('python analysis/plot_geometry_repind.py --results_dir results/geometry_repind_rco', cwd=CODE_DIR)
else:
    run('python analysis/plot_geometry_repind.py', cwd=CODE_DIR)

# Prompt attack type probe figures (if probe results exist)
probe_results = CODE_DIR / 'results/probe_attack_types/results.json'
if probe_results.exists():
    run('python analysis/plot_attack_types.py', cwd=CODE_DIR)

# Sync all figures to docs/figures/ and print inventory
run('python scripts/sync_figures.py', cwd=PROJECT)
run('python scripts/inventory.py', cwd=PROJECT)


## 12. Package Results

Create `colab_results.zip` with JSON results, report figures, direction tensors, probe prompts, and DIM metadata. Modified model weights are intentionally excluded because they are too large for direct notebook download.


In [ ]:
import shutil, zipfile
from pathlib import Path

RESULTS = PROJECT / 'code' / 'results'
DOCS_FIGS = PROJECT / 'docs' / 'figures'
DIM_RUN = PROJECT / 'code' / 'methods' / 'dim' / 'pipeline' / 'runs' / MODEL_ALIAS
DIM_DIRECTION = DIM_RUN / 'direction.pt'
RCO_DIRECTION = RESULTS / 'geometry_repind' / 'rco_direction.pt'
ZIP_PATH = PROJECT / 'colab_results.zip'

# Include the raw 2-D cone basis when available.
RCO_ROOT = PROJECT / 'code' / 'methods' / 'cones-repind' / 'results' / 'rdo'
rco_candidates = list(RCO_ROOT.glob('*/local_vectors/*/*/lowest_loss_vector.pt'))
RCO_BASIS = max(rco_candidates, key=lambda p: p.stat().st_mtime) if rco_candidates else None

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    # All result JSONs (benchmark, overlap, repind, probe, etc.)
    for json_file in sorted(RESULTS.rglob('*.json')):
        zf.write(json_file, f'results/{json_file.relative_to(RESULTS)}')
        print(f'  + results/{json_file.relative_to(RESULTS)}')

    # All synced figures
    if DOCS_FIGS.exists():
        for fig in sorted(DOCS_FIGS.glob('*.png')):
            zf.write(fig, f'figures/{fig.name}')
            print(f'  + figures/{fig.name}')

    # Per-experiment figures (sync_figures.py may not have them all)
    for fig in sorted(RESULTS.rglob('*.png')):
        zf.write(fig, f'results/{fig.relative_to(RESULTS)}')

    # DIM artifacts
    dim_meta = DIM_RUN / 'direction_metadata.json'
    if dim_meta.exists():
        zf.write(dim_meta, 'dim_metadata/direction_metadata.json')
        print(f'  + dim_metadata/direction_metadata.json')
    if DIM_DIRECTION.exists():
        zf.write(DIM_DIRECTION, 'directions/dim_direction.pt')
        print(f'  + directions/dim_direction.pt')

    # RCO direction (the standalone .pt extracted by run_rco_eval.sh)
    if RCO_DIRECTION.exists():
        zf.write(RCO_DIRECTION, 'directions/rco_direction.pt')
        print(f'  + directions/rco_direction.pt')

    # RCO raw 2-D cone basis (lowest_loss_vector.pt)
    if RCO_BASIS is not None:
        zf.write(RCO_BASIS, 'directions/rco_lowest_loss_vector.pt')
        print(f'  + directions/rco_lowest_loss_vector.pt  ({RCO_BASIS.relative_to(PROJECT)})')

print(f'\nZip created: {ZIP_PATH}')
print(f'Size: {ZIP_PATH.stat().st_size / 1024 / 1024:.1f} MB')

# Download in Colab
try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except ImportError:
    print('Not in Colab, skipping download. Zip is at:', ZIP_PATH)


## 13. Optional: Save Modified Models to Google Drive

Use this only if you need the full DIM, ActSVD, or RCO modified-model directories. These artifacts are large, so the default is to leave them in the Colab workspace and package only lightweight results in step 12.


In [ ]:
SAVE_MODELS_TO_DRIVE = False  # flip to True to copy modified models to Google Drive

if SAVE_MODELS_TO_DRIVE:
    import shutil
    from pathlib import Path

    try:
        from google.colab import drive as _drive
        _drive.mount('/content/drive')
    except Exception as exc:
        raise RuntimeError(f'Google Drive mount failed: {exc}')

    DRIVE_ROOT = Path('/content/drive/MyDrive/nlp-project-models') / MODEL_ALIAS
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    # Map of source dir -> destination subdir (only includes ones that exist)
    candidates = {
        'dim_modified':    PROJECT / 'code/methods/dim/pipeline/runs' / MODEL_ALIAS / 'modified_model',
        'actsvd_modified': PROJECT / 'code/methods/actsvd/out',
        'rco_modified':    PROJECT / 'code/methods/cones-repind/results/modified_model' / f'{MODEL_ALIAS}-RCO',
    }

    def _dir_size(p: Path) -> int:
        return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())

    for name, src in candidates.items():
        if not src.exists():
            print(f'[skip] {name}: not found at {src}')
            continue
        dst = DRIVE_ROOT / name
        size_gb = _dir_size(src) / 1024**3
        print(f'[copy] {name}: {size_gb:.1f} GB -> {dst}')
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'       done.')

    # Also stash the small artifacts (directions + metadata) alongside, so the
    # Drive folder is self-contained for offline analysis.
    aux = DRIVE_ROOT / 'aux'
    aux.mkdir(exist_ok=True)
    for src_path, arc_name in [
        (PROJECT / 'code/methods/dim/pipeline/runs' / MODEL_ALIAS / 'direction.pt',          'dim_direction.pt'),
        (PROJECT / 'code/methods/dim/pipeline/runs' / MODEL_ALIAS / 'direction_metadata.json','dim_direction_metadata.json'),
        (PROJECT / 'code/results/geometry_repind/rco_direction.pt',                          'rco_direction.pt'),
    ]:
        if src_path.exists():
            shutil.copy2(src_path, aux / arc_name)
            print(f'  + aux/{arc_name}')

    print(f'\nAll modified-model artifacts under: {DRIVE_ROOT}')
    print('Sync to your laptop via the Google Drive desktop app, gdrive CLI, or rclone.')
else:
    print('SAVE_MODELS_TO_DRIVE is False — skipping. Flip to True if you want the modified models locally.')
